# Infer independent context and population scores

This CPU walkthrough reads persisted train, validation, and test shards. The normal reference bank is fit from the persisted normal-only train split; validation and test labels are not used for fitting.

Set `V1_DATA_ROOT` to a generated dataset root, or use the stable default `data/generated/production`.

In [ ]:
from itertools import islice
import json
import os
from pathlib import Path
import sys
for candidate in (Path.cwd() / 'src', Path.cwd().parent / 'src'):
    if (candidate / 'representation').is_dir():
        sys.path.insert(0, str(candidate))
        break
from representation import V1Config
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference
from representation.model import V1RepresentationModel
from synth.config import PatchConfig
from synth.patchify import Patchifier

In [ ]:
configured_root = Path(os.environ.get('V1_DATA_ROOT', 'data/generated/production')).expanduser()
repo_root = Path.cwd()
if not (repo_root / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
DATA_ROOT = configured_root if configured_root.is_absolute() else repo_root / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Run uv run python -m synth.cli --output {DATA_ROOT} first.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")
def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples
train_samples = load_split('train')
val_samples = load_split('val', limit=2)
test_samples = load_split('test', limit=2)
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples], 'test IDs', [sample.file_id for sample in test_samples])

In [ ]:
cfg = V1Config(n_channels=train_samples[0].C, patch_size=32, stride=16, d_model=8, sequence_layers=1, attention_heads=2, dropout=0.0)
patchifier = Patchifier(PatchConfig(patch_size=32, stride=16, pad_end=True))
reference_batch = collate_variable_files(train_samples, patchifier, masking_config=cfg, masking_seed=5)
val_batch = collate_variable_files(val_samples, patchifier, masking_config=cfg, masking_seed=6)
test_batch = collate_variable_files(test_samples, patchifier, masking_config=cfg, masking_seed=7)
print('reference signals', tuple(reference_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'test signals', tuple(test_batch['signals'].shape))

In [ ]:
model = V1RepresentationModel(cfg, patchifier=patchifier).eval()
reference_output = model(reference_batch)
bank = NormalReferenceBank(k=min(2, len(train_samples))).fit(reference_output['file_embedding'])
inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)
for split, batch in (('val', val_batch), ('test', test_batch)):
    scores = inference.score_batch(batch)
    print(split, 'S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())
print('normal train reference rows', bank.embeddings.shape[0], 'labels were not passed to bank.fit')